In [1]:
# --- imports

# dash application and plotting
import dash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output, State
import dash_bootstrap_components as dbc
import plotly.express as px

# data management
import pandas as pd

# dbms
from neo4j import GraphDatabase

# misc
from datetime import date, datetime, timedelta
from configparser import ConfigParser

In [2]:
# --- configuration

# Read data from a configuration file and store in a dictionary
# as key-value pairs.
#
# Parameters:
#   filename: The name of the file that contains the configuration data.
#   section:  The name of the section within the configuration file
#             for which we are retrieving data.
# Returns: A dictionary with the key-value pairs from the configuration file.
def config(filename, section):
    cp = ConfigParser()
    # acquire data
    cp.read(filename)
    
    # convert to dictionary
    block = {}
    # make sure each section exists
    if cp.has_section(section):
        # read the entire section
        items = cp.items(section)
        # loop through config items to create the dictionary
        for item in items:
            block[item[0]] = item[1]
    else:
        # a section is missing, error out
        raise Exception(f"Section {section} was not found in the {filename} file!")
    return block

# Parse the provided config file as display results. Useful for testing.
# DEBUG ONLY, should not be called in production as it shows password!
#
# Parameters:
#   filename: The name of the file that contains the configuration data.
#   section:  The name of the section within the configuration file
#             for which we are retrieving data.
def displayConfig(filename, section):
    # obtain the config
    confData = config(filename, section)
    # display conf data (DEBUG ONLY, should not be called in production as it shows password!)
    print(confData)

In [3]:
# --- database connection

FILENAME = "database.conf"
SECTION="neo4j"

# Connect to the Neo4j database.
#
# Parameters:
#   filename: The name of the file that contains the configuration data.
#   section:  The name of the section within the configuration file
#             for which we are retrieving data.
# Returns: a Connection to the databse or null in the event of a problem.
def dbConnect(filename, section):
    # obtain configuration data from the config file
    confData = config(filename, section)
    
    try:     
        # Setup the connection to the Neo4j server
        # URI = "neo4j://" + confData["host"] + ":7687/" + confData["database"]
        # locally hosted on custom port as opposed to running public instance
        URI = "bolt://" + confData["host"] + ":" + confData["port"] + "/" + confData["database"]
        AUTH = (confData["user"],confData["password"])
        
        conn = GraphDatabase.driver(URI,auth=AUTH)
        return conn
    except(Exception) as error:
        # had a problem!
        print(f"ERROR: failed to connect to database: {error}")
        return None
    
# Utility method for testing the database connection.
# 
# Parameters:
#   filename: The name of the file that contains the configuration data.
#   section:  The name of the section within the configuration file
#             for which we are retrieving data.
# Returns: True if connection is successfully verifed, False if not.
def dbTest(filename, section):
    try:     
        # create our connection
        print("Connecting to our Neo4j database...")
        conn = dbConnect(filename, section)
    
        conn.verify_connectivity()
        print("Connection Verified!")
        conn.close()
        return True
    except(Exception) as error:
        # had a problem!
        print(f"ERROR: failed to verify connection to database: {error}")
        return False

In [4]:
# --- db client utilities

# readRecord - CRUD Method used to read a record from a database. 
# This method will be used to read a node from the graph.
# Because Neo4j is a graph database, this operation is specific
# to graph nodes and edges.
#
# Parameters:
#     query: cypher query used to return one or more records
#     conn: The database connection to use create the record
#    
def readRecord(query, conn):
    try:
        records, summary, keys = conn.execute_query(query, database_=db)
        
        # display success
        print(f"Successfully read {len(records)} record(s) from {db}!")
        return records
    except Exception as error:
        # something went wrong!
        print(f"ERROR: failed to read record(s) from {db}!")
        print(error)


#
# testReadRecord - Method used to test the readRecord function.
# This function will exercise the readRecord function by attempting to read
# a specific NutDes (Nutrient Description) node. This function will
# only work with the Nutrient database, it is not portable.
#
# Parameters:
#   filename: The name of the file that contains the configuration data
#   section:  The name of the section within the configuration file
#             for which we are retrieving data
#
def testReadRecord(filename, section):
    # create our connection
    print('Connecting to our Neo4j database...')
    conn = dbConnect(filename, section)
    
    # Setup dictionary to find data
    query = 'MATCH (n:NutDes) where n.NutrientCode = 777 return n'

    # Find records if they exist
    records = readRecord(query, conn)
    
    # Display results
    print('Records Retrieved:')
    if records is not None:
        for i in records:
            print(i)
    else:
        print('There were no records retrieved.')

    # Cleanup and close connection
    conn.close()

# Main cypher query for retrieving data for the Metrics Over Time tab.
#
# Parameters:
#  bucket: how long each time "bucket" is (daily, weekly, monthly)
#  tags_list: list of tags to filter by.
# Returns: pandas DataFrame
def get_metrics_data(bucket, tags_list):
    conn = dbConnect(FILENAME, SECTION)
    
    query = """
    // parameters: $bucket (string: 'day', 'week', 'month'), $tags (list of strings)
    MATCH (:User)-[r:POSTS]->(p)
    WHERE p:Question OR p:Answer OR p:Comment
    // only filter by tags if the list is not empty
    AND ($tags IS NULL OR size($tags) = 0 
         OR (p:Question AND EXISTS { (p)-[:TAGGED_WITH]->(t:Tag) WHERE t.name IN $tags })
         OR (p:Answer AND EXISTS { (p)-[:ANSWERS]->(:Question)-[:TAGGED_WITH]->(t:Tag) WHERE t.name IN $tags })
         OR (p:Comment AND EXISTS { (p)-[:RESPONDS_TO]->(:Question)-[:TAGGED_WITH]->(t:Tag) WHERE t.name IN $tags }))
    
    // determine the time bucket
    WITH p,
         CASE 
            WHEN $bucket = 'day' THEN date(r.creation_date)
            WHEN $bucket = 'week' THEN date(r.creation_date).year + '-W' + date(r.creation_date).week
            WHEN $bucket = 'month' THEN date(r.creation_date).year + '-' + date(r.creation_date).month
         END AS timeBucket,
         labels(p)[0] AS postType // Assumes the first label is the type
    
    // Aggregate counts
    RETURN timeBucket, postType, count(p) AS count
    ORDER BY timeBucket ASC;
    """

    with conn.session() as session:
        result = session.run(query, bucket=bucket, tags=tags_list)
        return pd.DataFrame([r.values() for r in result], columns=['timeBucket', 'postType', 'count'])

# Return a list of tags matching the search criteria.
#
# Parameters:
#  search_term: the string to search for.
# Returns: list of tag names as a string
def get_filtered_tags(search_term=""):
    conn = dbConnect(FILENAME, SECTION)
    query = """
    MATCH (t:Tag)
    WHERE $search = "" OR t.name CONTAINS $search
    RETURN t.name AS label, t.name AS value
    ORDER BY t.name
    """
    with conn.session() as session:
        result = session.run(query, {"search": search_term})
        return [r.value() for r in result]

In [5]:
# init with bootstrap styling
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# verify we can connect to the database
if not dbTest(FILENAME, SECTION):
    print("FATAL ERROR: connection to database failed!")
    raise

Connecting to our Neo4j database...
Connection Verified!


In [6]:
# -- create app header

# left: version number
VERSION_NUM = "v1.0.0"
header_version = dbc.Col(
    html.Span(VERSION_NUM, className="text-muted fw-bold"), 
    width=3, 
    className="d-flex align-items-center"
)
# center: application title
header_title = dbc.Col(
    html.H3("Stack Overflow Insights", className="text-center mb-0"), 
    width=6
)
# right: date and time display
header_time = dbc.Col(
    html.Span(id="live-clock", className="text-muted"), 
    width=3, 
    className="d-flex align-items-center justify-content-end"
)

# put together the header row
header_section = dbc.Row(
    [header_version, header_title, header_time],
    className="py-3 border-bottom mb-4 bg-light shadow-sm align-items-center"
)

In [7]:
# --- main tabs

# metrics over time

# get all tags to prepopulate the list
tags = get_filtered_tags("")
tags_filter = dbc.Col([
    html.Label("Search Tags:"),
    dcc.Input(id='tag-search-input', type='text', placeholder="Type to filter...", className="mb-2"),
    html.Div(
        dbc.Checklist(id='tag-checklist', options=tags, value=tags, switch=True),
        style={'height': '300px', 'overflowY': 'scroll', 'border': '1px solid #ccc', 'padding': '10px'}
    )
])

sidebar_controls = dbc.Col([
    tags_filter,
    html.Label("Bucket Size:", style={'marginTop': '20px'}),
    dcc.RadioItems(
        id='bucket-radio',
        options=[
            {'label': 'Day', 'value': 'day'}, 
            {'label': 'Week', 'value': 'week'}, 
            {'label': 'Month', 'value': 'month'}
        ],
        value='month'
    )
], width=12)

tab1_content = html.Div([
    dbc.Row([
        dbc.Col(
            dcc.Graph(id='metrics-time-chart'), 
            width=9
        ),
        dbc.Col(
            sidebar_controls, 
            width=3
        )
    ])
], className="p-4 border border-top-0 bg-white")

tab2_content = html.Div(
    html.H5("Search Database: Datatables and filters will go here.", className="text-center text-muted mt-5"),
    className="p-4 border border-top-0 bg-white"
)
tab3_content = html.Div(
    html.H5("Static Charts: Top/Bottom correlations will go here.", className="text-center text-muted mt-5"),
    className="p-4 border border-top-0 bg-white"
)

# assemble the tabs
main_tabs = dbc.Tabs(
    [
        dbc.Tab(tab1_content, label="Metrics Over Time", tab_id="tab-1"),
        dbc.Tab(tab2_content, label="Search Database", tab_id="tab-2"),
        dbc.Tab(tab3_content, label="Static Charts", tab_id="tab-3"),
    ],
    id="tabs",
    active_tab="tab-1", # sets the default active tab
    className="nav-fill" # spreads the tabs evenly across the width
)

main_content = html.Div(id="main-body", children=[main_tabs])

In [8]:
# --- main layout
app.layout = dbc.Container([
    # hidden interval component to trigger clock updates every 1000ms (1 second)
    dcc.Interval(id='clock-interval', interval=1000, n_intervals=0),
    header_section,
    main_content
], fluid=True, style={'minHeight': '100vh'})

In [9]:
# --- callbacks for updating content

# date time in the header
@app.callback(
    Output("live-clock", "children"),
    Input("clock-interval", "n_intervals")
)
def update_time(n):
    # formats to: "HH:MM:SS AM/PM | Month DD, YYYY"
    return datetime.now().strftime("%I:%M:%S %p | %B %d, %Y")

# update tags on search
@app.callback(
    Output('tag-checklist', 'options'),
    Input('tag-search-input', 'value')
)
def update_tag_list(search_term):
    # this only updates the dropdown/checklist choices based on the search
    return get_filtered_tags(search_term or "")

# update entire chart
@app.callback(
    Output('metrics-time-chart', 'figure'),
    [Input('tag-checklist', 'value'), 
     Input('bucket-radio', 'value')]
)
def update_metrics_chart(tags, bucket):
    # fetch data
    df = get_metrics_data(bucket, tags)
        
    if df.empty:
        # handle no data found
        return px.bar(title="No data available for selected criteria")

    # make chart
    fig = px.bar(
        df, 
        x='timeBucket', 
        y='count', 
        color='postType', 
        barmode='stack',
        category_orders={
            "postType": ["Question", "Answer", "Comment"]
        },
        color_discrete_map={
            "Question": "#BAC8D3", 
            "Answer": "#FAD9D5",   
            "Comment": "#B0E3E6"  
        }
    )

    fig.update_layout(
        xaxis_title="Time Period",
        yaxis_title="Total Count",
        legend_title="Post Type",
        template="plotly_white"
    )
    
    return fig

In [10]:
# --- run it!
if __name__ == '__main__':
    app.run(jupyter_mode="external", port=8050)

Dash app running on http://127.0.0.1:8050/
